# 01 — EDA: training-outcome panel + selection-on-observables structure

**Project:** Training ROI Predictor (H08) · **Author:** Asad Kamran · MADS, University of Michigan · Dubai HR

The panel is a synthetic 5,000-row employee × training corpus with a *known* ground-truth conditional average treatment effect. We profile (a) the treatment / outcome distributions, (b) the propensity-of-treatment structure, and (c) the gap between the naive ATE estimate and the truth — so the X-learner in `03_model.ipynb` has an audit baseline.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, str(Path.cwd().parent / 'src'))
from training_roi.data import (
    PROCESSED, DEPTS, TRAINING_TYPES, load_panel, make_training_artifacts,
)

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.dpi'] = 110

In [ ]:
PARQUET = PROCESSED / 'training_outcomes.parquet'
if PARQUET.exists():
    df = load_panel()
else:
    df = make_training_artifacts()
df.shape, df.columns.tolist()

## 1. Naive ATE vs ground truth

The naive estimator is the difference of treated and control mean uplift. Because treatment is selected on observables, the naive estimator is biased — that bias is the headline reason an X-learner exists.

In [ ]:
naive_ate = df.loc[df.treatment == 1, 'perf_uplift'].mean() - df.loc[df.treatment == 0, 'perf_uplift'].mean()
true_ate = df['true_cate'].mean()
print(f'naive ATE: {naive_ate:.3f}')
print(f'true ATE (mean true_cate): {true_ate:.3f}')
print(f'naive bias: {naive_ate - true_ate:.3f}')

## 2. Treatment propensity by `pre_perf` and `role_level`

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
for ax, col in zip(axes, ['pre_perf', 'role_level']):
    g = df.groupby(pd.cut(df[col], bins=5) if col == 'pre_perf' else df[col])['treatment'].mean()
    g.plot(kind='bar', ax=ax, color='#3a7ca5')
    ax.set_title(f'P(treated | {col})')
    ax.tick_params(axis='x', rotation=20)
plt.tight_layout(); plt.show()

## 3. Pre / post performance distributions

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.6))
sns.kdeplot(df, x='pre_perf', label='pre', ax=ax)
sns.kdeplot(df, x='post_perf', label='post', ax=ax)
ax.set_title('Pre vs post performance density')
ax.legend()
plt.tight_layout(); plt.show()

## 4. True CATE distribution

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.4))
sns.histplot(df, x='true_cate', bins=40, ax=ax, color='#9c6644')
ax.axvline(true_ate, color='red', ls='--', label=f'mean={true_ate:.2f}')
ax.set_title('Ground-truth CATE distribution (heterogeneity preview)')
ax.legend()
plt.tight_layout(); plt.show()

## 5. CATE by training kind

In [ ]:
kind_lookup = {tid: kind for tid, kind in TRAINING_TYPES}
df['training_kind'] = df['training_id'].map(kind_lookup)
fig, ax = plt.subplots(figsize=(7, 3.4))
sns.boxplot(data=df, x='training_kind', y='true_cate', ax=ax)
ax.set_title('CATE distribution by training kind')
plt.tight_layout(); plt.show()

## 6. CATE × pre_perf interaction (skill training)

In [ ]:
skill_only = df[df['training_kind'] == 'skill']
fig, ax = plt.subplots(figsize=(8, 3.6))
ax.scatter(skill_only['pre_perf'], skill_only['true_cate'], alpha=0.4, s=12, color='#3a7ca5')
ax.set_xlabel('pre_perf'); ax.set_ylabel('true CATE')
ax.set_title('Skill training — CATE vs pre_perf (lower pre → bigger uplift)')
plt.tight_layout(); plt.show()

## 7. CATE × role_level interaction (leadership training)

In [ ]:
lead_only = df[df['training_kind'] == 'leadership']
fig, ax = plt.subplots(figsize=(8, 3.4))
sns.boxplot(data=lead_only, x='role_level', y='true_cate', ax=ax)
ax.set_title('Leadership training — CATE by role_level')
plt.tight_layout(); plt.show()

## 8. Hypotheses for `03_model.ipynb`

1. **The X-learner will recover a per-employee CATE with MAE small enough to identify the top-quintile gainers.**
2. **The propensity-blend term is critical** — without it, the bias from selection-on-observables remains.
3. **Bootstrap CIs will be wide enough at the per-employee level** that the dashboard must show the band, not the point alone.